# Random Forest Classifier

This notebook trains and evaluates a Random Forest classifier for the CICIDS2017 multi-class network attack classification task.

The same datasets are used as in the other notebooks.

Random Forest is an ensemble method that trains multiple decision trees and combines their predictions. This usually makes it more stable than a single Decision Tree.

Load data

In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv("../train_data.csv")
validation = pd.read_csv("../val_data.csv")
test = pd.read_csv("../test_data.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

print(train["Attack Type"].value_counts())

Train: (60000, 11)
Validation: (20000, 11)
Test: (20000, 11)
Attack Type
Normal Traffic    49849
DoS                4604
DDoS               3069
Port Scanning      2144
Brute Force         226
Web Attacks          55
Bots                 53
Name: count, dtype: int64


Features and label

In [2]:
LABEL_COL = "Attack Type"

FEATURE_COLS = [col for col in train.columns if col != LABEL_COL]

train = train.replace([np.inf, -np.inf], np.nan)
validation = validation.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

train_x = train[FEATURE_COLS]
train_y = train[LABEL_COL]

validation_x = validation[FEATURE_COLS]
validation_y = validation[LABEL_COL]

test_x = test[FEATURE_COLS]
test_y = test[LABEL_COL]

print("Features:")
print(FEATURE_COLS)
print("Number of features:", len(FEATURE_COLS))
print("Classes:", train_y.unique())

Features:
['Bwd Packet Length Std', 'Subflow Fwd Bytes', 'Flow Duration', 'Total Length of Fwd Packets', 'Init_Win_bytes_forward', 'Flow IAT Std', 'Active Mean', 'Bwd Packets/s', 'Fwd Packet Length Mean', 'Bwd Packet Length Min']
Number of features: 10
Classes: <StringArray>
[          'DDoS', 'Normal Traffic',  'Port Scanning',            'DoS',
    'Brute Force',    'Web Attacks',           'Bots']
Length: 7, dtype: str


Import libraries
- RandomForestClassifier is the model.
- SimpleImputer handles missing values.
- StandardScaler and MinMaxScaler are included for combinations that use SMOTE, PCA or LDA.
- SMOTE is included because the attack classes are imbalanced.
- PCA and LDA are tested as optional dimensionality reduction methods.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    make_scorer
)
from sklearn.model_selection import GridSearchCV, PredefinedSplit, ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

Train, validation and test datasets are already created, so PredefinedSplit is used.
- GridSearchCV trains on the training rows.
- GridSearchCV validates on the validation rows.
- The separate test set is only used after model selection.

In [4]:
# Combine train and validation sets for GridSearchCV
train_val_x = pd.concat([train_x, validation_x], axis=0)
train_val_y = pd.concat([train_y, validation_y], axis=0)

# Use the original validation set as the validation fold.
# -1 = training rows, 0 = validation rows
validation_fold = [-1] * len(train_x) + [0] * len(validation_x)
validation_split = PredefinedSplit(validation_fold)

Pipeline introduced
- Imputation
- Optional scaling
- Optional SMOTE
- Optional dimensionality reduction
- Random Forest model

In [5]:
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", "passthrough"),
    ("smote", "passthrough"),
    ("dimred", "passthrough"),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

Parameter grid defines different combinations
- Basic Random Forest without scaling, SMOTE or dimensionality reduction
- Random Forest with SMOTE
- Random Forest with PCA or LDA
- Class weighting is tested because the dataset is imbalanced

In [6]:
rf_param_grid = [
    # Basic Random Forest without SMOTE or dimensionality reduction
    {
        "scaler": ["passthrough"],
        "smote": ["passthrough"],
        "dimred": ["passthrough"],
        "model__n_estimators": [100],
        "model__max_depth": [None, 20],
        "model__min_samples_leaf": [1, 3],
        "model__max_features": ["sqrt"],
        "model__class_weight": [None, "balanced_subsample"]
    },

    # Random Forest with SMOTE
    {
        "scaler": [StandardScaler(), MinMaxScaler()],
        "smote": [SMOTE(random_state=RANDOM_STATE, k_neighbors=3)],
        "dimred": ["passthrough"],
        "model__n_estimators": [100],
        "model__max_depth": [20],
        "model__min_samples_leaf": [1, 3],
        "model__max_features": ["sqrt"],
        "model__class_weight": [None]
    },

    # Random Forest with dimensionality reduction
    {
        "scaler": [StandardScaler()],
        "smote": ["passthrough"],
        "dimred": [
            PCA(n_components=5, random_state=RANDOM_STATE),
            LDA(n_components=2)
        ],
        "model__n_estimators": [100],
        "model__max_depth": [20],
        "model__min_samples_leaf": [1, 3],
        "model__max_features": ["sqrt"],
        "model__class_weight": [None, "balanced_subsample"]
    }
]

# Define scoring metrics
scoring = {
    "accuracy": "accuracy",
    "macro_precision": make_scorer(precision_score, average="macro", zero_division=0),
    "macro_recall": make_scorer(recall_score, average="macro", zero_division=0),
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0)
}

number_of_combinations = sum(len(ParameterGrid(grid)) for grid in rf_param_grid)
print("Number of tested parameter combinations:", number_of_combinations)

Number of tested parameter combinations: 20


GridSearchCV trains and evaluates all parameter combinations
- The validation fold is fixed with PredefinedSplit.
- `refit="accuracy"` means the best parameter combination is refitted after validation scoring.
- Multiple metrics are stored so accuracy and macro scores can be compared.

In [ ]:
# Perform GridSearchCV
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring=scoring,
    refit="accuracy",
    cv=validation_split,
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
    error_score="raise"
)

# Fit the model
grid_search.fit(train_val_x, train_val_y)

Print best validation parameters and validation accuracy

In [ ]:
print("Best Validation Parameters:", grid_search.best_params_)
print("Best Validation Accuracy:", grid_search.best_score_)

Best Validation Parameters: {'dimred': 'passthrough', 'model__class_weight': None, 'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__n_estimators': 100, 'scaler': 'passthrough', 'smote': 'passthrough'}
Best Validation Accuracy: 0.9967


GridSearchCV results table for all tested combinations
- Validation accuracy
- Validation macro precision, recall and F1
- Training accuracy
- Forest and preprocessing parameters

In [ ]:
rf_results_table = pd.DataFrame(grid_search.cv_results_)

columns_to_display = [
    "rank_test_accuracy",
    "param_scaler",
    "param_smote",
    "param_dimred",
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "param_model__class_weight",
    "mean_test_accuracy",
    "mean_test_macro_precision",
    "mean_test_macro_recall",
    "mean_test_macro_f1",
    "mean_train_accuracy"
]

rf_results_table_display = rf_results_table[columns_to_display].sort_values(
    by="rank_test_accuracy"
).rename(columns={
    "rank_test_accuracy": "Rank",
    "param_scaler": "Scaler",
    "param_smote": "SMOTE",
    "param_dimred": "Dimensionality Reduction",
    "param_model__n_estimators": "Number of Trees",
    "param_model__max_depth": "Max Depth",
    "param_model__min_samples_leaf": "Min Samples Leaf",
    "param_model__max_features": "Max Features",
    "param_model__class_weight": "Class Weight",
    "mean_test_accuracy": "Validation Accuracy",
    "mean_test_macro_precision": "Validation Macro Precision",
    "mean_test_macro_recall": "Validation Macro Recall",
    "mean_test_macro_f1": "Validation Macro F1",
    "mean_train_accuracy": "Train Accuracy"
})

rf_results_table_display

,Rank,Scaler,SMOTE,Dimensionality Reduction,Number of Trees,Max Depth,Min Samples Leaf,Max Features,Class Weight,Validation Accuracy,Validation Macro Precision,Validation Macro Recall,Validation Macro F1,Train Accuracy
0,1,passthrough,passthrough,passthrough,100,None,1,sqrt,NaN,0.99670,0.932752,0.841084,0.874157,0.999583
2,2,passthrough,passthrough,passthrough,100,20,1,sqrt,NaN,0.99660,0.843592,0.801560,0.820272,0.998967
6,3,passthrough,passthrough,passthrough,100,20,1,sqrt,balanced_subsample,0.99650,0.913916,0.828997,0.862565,0.999583
4,3,passthrough,passthrough,passthrough,100,None,1,sqrt,balanced_subsample,0.99650,0.921821,0.828828,0.862577,0.999583
1,5,passthrough,passthrough,passthrough,100,None,3,sqrt,NaN,0.99570,0.822598,0.756715,0.781288,0.997217
5,6,passthrough,passthrough,passthrough,100,None,3,sqrt,balanced_subsample,0.99560,0.853448,0.925512,0.878185,0.998083
3,7,passthrough,passthrough,passthrough,100,20,3,sqrt,NaN,0.99550,0.818594,0.748032,0.772777,0.997167
7,8,passthrough,passthrough,passthrough,100,20,3,sqrt,balanced_subsample,0.99530,0.850217,0.925376,0.875184,0.997967
8,9,StandardScaler(),"SMOTE(k_neighbors=3, random_state=42)",passthrough,100,20,1,sqrt,NaN,0.99350,0.834167,0.964873,0.862700,0.995000
9,10,MinMaxScaler(),"SMOTE(k_neighbors=3, random_state=42)",passthrough,100,20,1,sqrt,NaN,0.99330,0.833484,0.966579,0.862783,0.994933


Best model with test data
- Selected with validation data.
- Evaluated with separate test data.
- Macro scores are included because minority classes are important in attack detection.

In [ ]:
# Evaluate the best model on the test set
best_rf_model = grid_search.best_estimator_

# Make predictions
train_predictions = best_rf_model.predict(train_x)
validation_predictions = best_rf_model.predict(validation_x)
test_predictions = best_rf_model.predict(test_x)

# Calculate test metrics
test_accuracy = accuracy_score(test_y, test_predictions)
test_macro_precision = precision_score(test_y, test_predictions, average="macro", zero_division=0)
test_macro_recall = recall_score(test_y, test_predictions, average="macro", zero_division=0)
test_macro_f1 = f1_score(test_y, test_predictions, average="macro", zero_division=0)

# Calculate train and validation accuracy after the final refit
train_accuracy = accuracy_score(train_y, train_predictions)
validation_accuracy_after_refit = accuracy_score(validation_y, validation_predictions)

# Print results
print("Train Accuracy:", train_accuracy)
print("Validation Accuracy from GridSearchCV:", grid_search.best_score_)
print("Validation Accuracy after final refit:", validation_accuracy_after_refit)
print("Test Accuracy:", test_accuracy)
print("Test macro Precision:", test_macro_precision)
print("Test macro Recall:", test_macro_recall)
print("Test macro F1:", test_macro_f1)

# Print confusion matrix and classification report
class_labels = best_rf_model.named_steps["model"].classes_

print("Class order:", class_labels)
print()
print("Confusion Matrix:")
print(confusion_matrix(test_y, test_predictions, labels=class_labels))
print()
print("Classification Report:")
print(classification_report(test_y, test_predictions, labels=class_labels, zero_division=0))

Train Accuracy: 0.9995666666666667
Validation Accuracy from GridSearchCV: 0.9967
Validation Accuracy after final refit: 0.9996
Test Accuracy: 0.99635
Test macro Precision: 0.910511055435517
Test macro Recall: 0.8610110955048685
Test macro F1: 0.8800466400226056
Class order: ['Bots' 'Brute Force' 'DDoS' 'DoS' 'Normal Traffic' 'Port Scanning'
 'Web Attacks']

Confusion Matrix:
[[   11     0     0     0     7     0     0]
 [    0    70     0     1     4     0     0]
 [    0     0  1022     0     1     0     0]
 [    0     0     0  1517    18     0     0]
 [    8     0     2    12 16585     7     2]
 [    0     0     1     0     1   713     0]
 [    0     0     0     0     9     0     9]]

Classification Report:
                precision    recall  f1-score   support

          Bots       0.58      0.61      0.59        18
   Brute Force       1.00      0.93      0.97        75
          DDoS       1.00      1.00      1.00      1023
           DoS       0.99      0.99      0.99      1535
N

Confusion matrix as a table
- Rows are true classes.
- Columns are predicted classes.
- This makes it easier to see which attack types are confused with each other.

In [ ]:
rf_confusion_matrix = pd.DataFrame(
    confusion_matrix(test_y, test_predictions, labels=class_labels),
    index=class_labels,
    columns=class_labels
)

rf_confusion_matrix

,Bots,Brute Force,DDoS,DoS,Normal Traffic,Port Scanning,Web Attacks
Bots,11,0,0,0,7,0,0
Brute Force,0,70,0,1,4,0,0
DDoS,0,0,1022,0,1,0,0
DoS,0,0,0,1517,18,0,0
Normal Traffic,8,0,2,12,16585,7,2
Port Scanning,0,0,1,0,1,713,0
Web Attacks,0,0,0,0,9,0,9


Feature importance
- Random Forest can show which input features were most useful across the trees.
- If PCA or LDA is selected, original feature importance is not directly interpretable.

In [ ]:
model_step = best_rf_model.named_steps["model"]
dimred_step = best_rf_model.named_steps["dimred"]

if dimred_step == "passthrough":
    rf_feature_importance = pd.DataFrame({
        "Feature": FEATURE_COLS,
        "Importance": model_step.feature_importances_
    }).sort_values("Importance", ascending=False)

    rf_feature_importance
else:
    print("Feature importance is not directly interpretable because the best model used PCA or LDA.")

Random Forest summary

Best Random Forest model is selected by validation accuracy.

In [ ]:
rf_summary = pd.DataFrame([{
    "Model": "Random Forest",
    "Best Parameters": grid_search.best_params_,
    "Train Accuracy": train_accuracy,
    "Validation Accuracy": grid_search.best_score_,
    "Validation Accuracy After Refit": validation_accuracy_after_refit,
    "Test Accuracy": test_accuracy,
    "Test Macro Precision": test_macro_precision,
    "Test Macro Recall": test_macro_recall,
    "Test Macro F1": test_macro_f1
}])

rf_summary

,Model,Best Parameters,Train Accuracy,Validation Accuracy,Validation Accuracy After Refit,Test Accuracy,Test Macro Precision,Test Macro Recall,Test Macro F1
0,Random Forest,"{'dimred': 'passthrough', 'model__class_weight...",0.999567,0.9967,0.9996,0.99635,0.910511,0.861011,0.880047
